# Offline vs. Online Evaluation
*Offline evaluation's three benefits, the three risks unique to production, what an online eval pipeline actually looks like, captured vs. computed signals, stratified sampling, and the self-improving loop between online and offline evals*

## Two ways an eval can run

Everything covered so far — the workflow, the reasons for multiple pipelines, the three eval methods — describes *how* an eval is built and *who* executes it. There's a separate, orthogonal question: *when* does it run, relative to deployment? That's the distinction between **offline evaluation** and **online evaluation**.

## What offline evaluation is

An offline eval is any evaluation pipeline run on an LLM-based application **before** it's deployed. Every worked example covered so far — the retriever's Recall@k, the chatbot's helpfulness rubric, the exam-grading judge — is an offline eval: each one evaluates a system that exists, but hasn't yet been shipped to real users. Nothing new needs to be learned to define it; it's simply the name for the category all of that prior work already falls into.

Put simply: whenever there's a finished LLM-based application and the goal is to thoroughly test it before deciding whether it's ready to deploy, the evals run at that stage are offline evals.

## Benefit 1: pre-release testing and deployment gating

The most direct benefit of offline evaluation is straightforward: it prevents shipping an LLM-based application into production without knowing how it behaves first. Deploying untested software carries real risk — the earlier case studies (a chatbot inventing a refund policy, a dealership's chatbot agreeing to sell a car for a dollar) are exactly what deploying without offline evaluation can lead to.

In a mature setup, this testing step gets automated as part of the deployment pipeline itself, the same way software engineering teams use CI/CD (continuous integration / continuous deployment). A typical version of this looks like: a code change gets pushed, it triggers an automated pipeline (through a tool like GitHub Actions), and that pipeline runs the eval script against the golden dataset. If the resulting score clears a defined threshold — say, 95% — the deployment pipeline proceeds automatically. If the score falls short, the deployment is blocked and a notification goes out instead. Offline evaluation, wired up this way, becomes a **release gate**: change is deployed only if it demonstrably passes, and blocked or rolled back to the previous version if it doesn't.

## Benefit 2: comparing versions

Offline evaluation also makes it possible to directly compare two candidate versions of a system before committing to either one. Suppose the only undecided variable in a system is which underlying LLM to use — one candidate model from one provider versus another candidate model from a different provider, with everything else in the application held constant. Since the same eval and the same golden dataset apply cleanly to both versions, running that identical eval against each version puts them on level footing, and whichever version scores higher is the one to ship.

This isn't limited to choosing between models — the same approach applies to comparing different prompts, different rerankers, different vector databases, or different architectures for the same application. Whenever there's a genuine choice between two or more implementations, running the same offline eval against each one is what settles it with evidence rather than guesswork.

## Benefit 3: regression testing

The third benefit is catching **regressions** — cases where fixing or improving one thing quietly breaks something else that was previously working fine.

Consider a chatbot already live in production, where user feedback reveals that its tone comes across as somewhat cold specifically when a user asks about refunds. To fix that, the system prompt gets an instruction added: respond kindly and politely. After that change, the chatbot's overall tone does become warmer — but it also starts softening or rounding off factual details it previously stated precisely. Asked the exact cost of a plan priced at 19,500, it might now say "around 19,000" instead of the exact figure, apparently trying to sound more agreeable. The original problem — tone on refund questions — got fixed, but a new problem was introduced elsewhere: numerical precision got quietly sacrificed for a friendlier-sounding response.

This is exactly what regression testing is built to catch. If the golden dataset is built to represent every category of question a system handles — refunds, pricing, general policy questions, and so on — then re-running the same offline eval after any change makes it possible to check each category's score individually, not just the overall average. Suppose the success rate on refund-related questions was 90% before a prompt change. After the change, that category should still land around 90% — if it instead drops to 80%, that's a signal of regression: something that used to work is now working less well, even if the specific issue the change was meant to fix did improve. Any change to a system — a new prompt, a new model, a new vector database, anything at all — should be checked this way: does the intended improvement actually show up, and does anything else get worse in the process?

## Why offline evaluation is indispensable

Put together, offline evaluation earns its place through three concrete benefits: it provides pre-release testing that can double as an automated deployment gate, it enables direct, apples-to-apples comparison between candidate versions of a system, and it catches regressions that a change elsewhere in the system might otherwise introduce unnoticed. Between these three, offline evaluation is what makes it possible to trust that a system is ready to ship — and to keep trusting that after every subsequent change.

## Three risks that only show up in production

Passing every offline eval and deploying successfully doesn't mean the risk is over. Once a system is live, it faces three major categories of risk that offline evaluation, by its nature, cannot cover.

### Risk 1: unanticipated inputs

A golden dataset built for offline testing might contain a few hundred carefully sampled questions — everything the team anticipates a user might ask. But deploying a system opens it up to the real world, which is a much larger superset of what any golden dataset can anticipate: users mixing languages mid-sentence, vague half-formed questions, messages that are really an angry rant with a question buried somewhere inside, and deliberate adversarial prompt injections trying to extract a response the system was explicitly built not to give. None of this can be fully anticipated in advance, no matter how carefully the offline dataset was built.

### Risk 2: emergent and systematic failures

Some failures only become visible at scale, and never show up in a small offline test set no matter how well-designed it is. Two examples make this concrete:

- **Load-driven failures.** A sudden spike in concurrent users — after a product launch, say — can push latency up in ways a small offline dataset could never surface, since offline testing never simulates thousands of simultaneous real users.
- **Subtle, statistical bias.** A system might respond slightly differently to users from non-technical backgrounds compared to technical ones — a pattern that only becomes visible once thousands of real conversations accumulate, far more than any offline dataset would ever contain.

### Risk 3: drift

The most technical of the three risks is **drift** — the gradual process by which an offline eval, built and validated at one point in time, quietly becomes outdated.

Consider a RAG chatbot built on top of a knowledge base — pricing pages, curriculum documents, policy pages — as they existed on the day the golden dataset was built. Over time, prices change, curricula get revised, and policies get updated, the way any live business's documentation naturally evolves. A year later, the underlying documents can look very different from what they were when the golden dataset was created — but the golden dataset itself hasn't changed to match. The eval pipeline built against the old data can keep reporting good scores when run offline, even as the system's real-world behavior — judged against real users' negative feedback — quietly gets worse, simply because what counts as a "correct" answer has moved out from under the eval that's supposed to be checking for it.

## Why offline evaluation can't cover these risks

All three risks share the same root cause: offline evaluation works by comparing a system's output against a golden dataset built in advance, with known-correct answers. Once a system is in production, that assumption breaks down — a real user can ask literally anything, there's no golden dataset covering questions nobody anticipated, and there's no way to know in advance what the correct answer to tomorrow's question will even be. Offline evaluation, no matter how thorough, simply has no mechanism for handling inputs and situations it was never built to anticipate.

## What online evaluation is

This is exactly the gap **online evaluation** exists to close: evaluating a system on live production traffic, after deployment, as real users interact with it. Its single most defining characteristic is that it operates **without an answer key** — there's no golden dataset behind it, because by definition it's evaluating traffic nobody built a golden dataset for in advance. That's also precisely why it matters: it's the only mechanism available for confirming that a deployed system keeps running the way it should, once real-world usage is the only kind of usage happening.

## Offline vs. online: the key differences

| Dimension | Offline evaluation | Online evaluation |
|---|---|---|
| **Timing** | Before deployment | After deployment, continuously |
| **Data** | A fixed, purpose-built dataset | Live production traffic |
| **Answer key** | A golden dataset with known-correct answers | No answer key — nothing is known in advance |
| **Input coverage** | Only anticipated inputs | Anything a real user might send |
| **What it catches** | Regressions | Drift, emergent bugs, and unanticipated surprises |
| **Cost and speed** | Fast, cheap, and repeatable — a script or LLM run against a small dataset | Can be costly at scale, since it may mean evaluating every real conversation |

That last row matters in practice: if a system handles 50,000 conversations, evaluating every single one online can get expensive. A common workaround is **sampling** — randomly evaluating a representative subset (say, 1,000 out of 50,000 conversations) rather than all of them, trading some coverage for a much lower cost.

## Complementary, not rivals

Offline and online evaluation aren't alternatives to choose between — they're complementary, and a properly evaluated system needs both running at once. The clearest way to state the difference: **offline evaluation checks whether an application is working correctly. Online evaluation checks whether it's running normally.** Those are two different questions, and only one of them is answerable once real, unlabeled production traffic is all there is to work with.

## Correctness vs. normality: a worked example

Recall the exam-grading judge from the earlier model-graded example. Offline, its correctness was defined very specifically: how close its marks landed to a human expert's marks on the same answers, using the golden dataset built for that purpose.

Once that system is deployed, can its correctness still be measured the same way? No — and the reason is direct: at the moment a new aspirant's answer comes in, there's no human-graded score for that specific answer to compare against. The golden dataset only covers the answers it was built from; it says nothing about an answer nobody has scored before. Correctness, in the strict sense used offline, simply isn't measurable anymore.

What *can* still be checked is whether the system is behaving normally — consistently with how it always has — using its own history as a baseline. Suppose every graded answer's total score gets logged, and a full week's worth of scores gets plotted as a distribution:

| Period | Where most scores cluster (out of 1000) |
|---|---|
| Weeks 1–4 (established baseline) | Around 500 |
| Week 5 | Around 800 |

If weeks 1 through 4 consistently cluster around the same range, that becomes the system's baseline. A sudden shift in week 5 — scores clustering much higher than that baseline — is a signal that something changed, even without knowing whether the new scores are themselves right or wrong. It might turn out to be a genuinely stronger batch of aspirants that week, or it might be a real problem with the grading system; either way, the shift is what triggers the investigation. That's the distinction in practice: online evaluation flagged that something was *different*, not that anything specific was *wrong* — normality, not correctness, is what it directly measures.

## Getting closer to correctness without an answer key

Some metrics don't actually require knowing the correct answer in the first place, which makes them usable online even without a golden dataset. **Faithfulness** in a RAG system is one: judging whether a generated answer is grounded in its retrieved context doesn't require knowing whether the underlying context itself was the right thing to retrieve — the context and the answer are both directly available, so an LLM can judge whether one is faithful to the other without any external reference at all.

Where a metric does fundamentally require a correct answer — like whether a chatbot's response was actually right — a proxy signal has to stand in instead. User feedback is the most direct one: if a chatbot has thumbs-up and thumbs-down controls built in, and thumbs-down ratings suddenly spike over a given hour, that's a strong signal that something is going wrong with the answers being given, even without knowing the specific correct answer to any of the underlying questions. It's an indirect, imperfect substitute for correctness — but in a setting with no answer key at all, it's often the closest available approximation.

## What an online eval pipeline actually looks like

With the concepts in place, the natural next question is mechanical: what does an online eval pipeline concretely consist of, step by step? It starts with a step that has to come before anything else can happen at all: **logging**.

## Step 1: logging

If a chatbot is live and handling real conversations, nothing about those conversations can be evaluated unless it's captured somewhere first — anything not recorded is simply gone. Logging means capturing a structured, replayable record of every conversation turn, which typically includes:

- A **conversation ID**, a **turn ID** for each back-and-forth exchange, a **user ID**, a **session ID**, and a **timestamp**.
- The **question** the user asked.
- For a RAG system, the **retrieved context** used to answer it.
- The **generated output** itself.
- Operational details: **latency** in milliseconds, **prompt and completion token counts**, **total cost**, and any **error status codes**.
- Behavioral signals: whether the user gave a **thumbs-up or thumbs-down**, whether the conversation got **escalated** (asking for a human, or providing an email for follow-up), or whether the user kept **rephrasing the same question**, suggesting the system wasn't actually helping.

A tool built specifically for this — an LLM observability platform like LangSmith — is typically used to store all of this in a structured, searchable way, the same way traditional software relies on logging and observability infrastructure.

## What good logging requires

A handful of engineering properties make logging actually usable downstream:

- **Non-blocking.** Logging is itself an operation running alongside the conversation, and it shouldn't add latency to it — the conversation and the logging happen in parallel, not one blocking the other.
- **Durable and queryable.** Data needs to land somewhere structured — a data warehouse or an observability tool — so it can be fetched and analyzed later, not just written once and forgotten.
- **Late signal attachment.** Some signals don't arrive during the conversation itself. If a user escalates a day later by emailing support, that escalation still needs to be tied back to the original conversation ID it relates to, even though it arrived well after the conversation ended.
- **PII handling.** Any sensitive personal information a user shares mid-conversation — a phone number, an address, a card number — needs to be masked or redacted before storage, so that nobody looking through the logs later can extract it.

## Two categories of online signals: captured vs. computed

Once logging is in place, the signals worth tracking in production split into two categories:

- **Captured signals** are already present in the raw data — they just need to be pulled out and stored as-is, with no calculation involved. A thumbs-up or thumbs-down, response latency, and token cost are all captured signals: the value already exists the moment the conversation happens.
- **Computed signals** require actually running something to derive them. Faithfulness, answer relevance, correctness, hallucination rate, toxicity, and bias and fairness all fall here — none of these come for free from the raw logs; each one needs an evaluator to calculate it.

## The pipeline for captured quantities: log, dashboard, alert

For a captured quantity, nothing needs to be computed — the flow is simply: **log it, put it on a dashboard, and set up alerts on it.**

A dashboard typically shows a quantity like latency aggregated over different time windows — the last hour, the last 24 hours, the last week, the last six months — because a single conversation in isolation rarely matters on its own; what matters is the trend. If latency ticks up for one user's one conversation, that's noise. If average latency across the last hour is climbing across many conversations, that's a signal something is systemically off.

A concrete scenario: a new product launches, and 500 users suddenly hit the chatbot at once. The dashboard's latency graph, previously flat, visibly climbs. That climb triggers an alert — routed to a channel like Slack, email, or an on-call paging tool — notifying the team that latency has crossed a defined threshold. The team responds by allocating more resources (spinning up additional server instances, adjusting a load balancer), and latency comes back down once the extra capacity absorbs the load. **Logging → dashboarding → alerting** is the complete pipeline for any captured quantity — no evaluator needs to be built at all.

## The pipeline for computed quantities: an added evaluation step

Computed quantities need one more step in the middle. Take **hallucination rate** as a concrete example: checking, in real time, whether a chatbot's answers are actually grounded or fabricated.

This is a **reference-free** evaluation — there's no golden dataset with a predefined "correct" answer for a live user's brand-new question, so there's nothing to compare against directly. The way to evaluate it anyway is with an **LLM-as-judge**: a separate, often more capable LLM is given the retrieved context, the user's question, and the original system's generated output, along with a detailed rubric describing what counts as a hallucination, and asked to judge whether the answer is grounded in the context it was given.

## Why sampling is necessary

Running that judge on every single conversation is prohibitively expensive. If a chatbot handles 500 conversations a day, the cost of running each one through an LLM judge is on top of the cost of running the chatbot itself — potentially more than doubling total cost. The practical fix is **sampling**: rather than evaluating all 500 conversations, a smaller number is randomly selected — say, 100 out of 500 — and the judge runs only on that subset. The resulting hallucination rate, computed from the sample, gets sent to the same dashboard as any captured quantity, and the same alerting logic applies on top of it.

## Random sampling vs. stratified sampling

Plain random sampling works, but it isn't the strongest choice available. Not every conversation is equally likely to contain a hallucination — conversations that ended with a thumbs-down, that were abandoned abruptly, that involved an escalation, that show the user rephrasing the same question repeatedly, or that touch on higher-stakes topics like refunds, fees, or admissions are all more likely to be where something actually went wrong.

**Stratified sampling** takes this into account directly: conversations are first divided into categories, and more samples are deliberately drawn from the higher-risk categories — thumbs-down conversations, escalations, money-related topics — while fewer are drawn from routine, seemingly successful ones. This does typically require a small classifier or simple detection model to sort conversations into these categories in the first place, but the payoff is a sampled set with a much better chance of actually surfacing hallucinations, rather than a sample that's just as likely to be full of unremarkable conversations.

## The full online eval pipeline, end to end

Putting the pieces together, an online eval pipeline for a computed quantity like hallucination rate looks like this: **log every conversation → sample from the logs (stratified, not purely random) → run an evaluator (an LLM-as-judge, for a reference-free metric like this one) → aggregate the resulting scores over a time window → display them on a dashboard → trigger alerts when a threshold is crossed.** For a captured quantity, the same pipeline applies with the sampling and evaluator steps skipped entirely, since nothing needs to be computed.

This is the shape of a single evaluation pipeline for a single metric. In practice, a production system runs several of these in parallel — one for latency, one for hallucination rate, one for toxicity, and so on — each with its own evaluator, its own sampling strategy where relevant, and its own dashboard and alerting thresholds, echoing the same point from earlier: one application ends up with several evaluation pipelines running against it, not just one.

## The same evaluator can run online or offline

An evaluation platform typically doesn't force a hard split between building an "online evaluator" and an "offline evaluator" as two different things. The same evaluator — the same rubric, the same underlying judge model, the same prompt — can be pointed at either of two targets:

- **Live traces** — the ongoing log of real conversations. Pointing an evaluator at this makes it an **online evaluator**, since it's running against production traffic as it happens.
- **A dataset** — a curated, pre-built set of examples. Pointing the same evaluator at this instead makes it an **offline evaluator**, since it's running against a fixed golden dataset rather than live traffic.

What determines whether a given evaluator counts as online or offline isn't anything about how it's built — it's simply what it's pointed at. The rubric checking for hallucination, for instance, can run against yesterday's live conversations one day and against a curated test dataset the next, and it's the same evaluator doing both jobs.

## The self-improving loop

This is what actually closes the circle between online and offline evaluation. When a live conversation is flagged as problematic — a hallucination caught by an online evaluator, a low rating from a user, a manually reviewed conversation that clearly went wrong — that specific conversation can be pulled directly out of the production logs and added to the offline golden dataset. Some platforms support this as a direct action on a logged conversation (an "add to dataset" option), sometimes paired with an annotation step where a person notes exactly what went right or wrong in that specific exchange.

Once added, that conversation becomes part of what the *next* offline evaluation run gets tested against — meaning the system now has to handle a real failure case it previously didn't, before it can be considered ready to redeploy. The full loop looks like this: build and offline-evaluate a system, deploy it, monitor it online, catch real failures, fold those specific failures back into the offline dataset, evaluate offline again against the now-richer dataset, and deploy again. Each cycle through this loop leaves the golden dataset a little more representative of what the system actually encounters in the real world — which is why offline and online evaluation are best understood as one continuous, self-reinforcing process rather than two separate activities that happen to share some infrastructure.

## Reading a dashboard against a baseline

A dashboard number only means something in relation to a baseline — and that baseline typically comes from the system's own offline evaluation results. Suppose a system's offline-evaluated faithfulness score was 0.85, and that becomes the baseline it gets compared against once deployed. If the live dashboard now shows faithfulness at 0.87, that's an improvement over baseline — a good sign. But if faithfulness suddenly drops to 0.75 over the past 24 hours, measured against that same 0.85 baseline, that's a meaningful regression worth investigating, and it's exactly the kind of shift that should trigger an alert rather than get noticed only by chance.

## A caution when comparing scores across a change

Seeing one metric improve — say, accuracy climbing from 92% to 99% after a change — isn't, by itself, enough to call the change a clear win. Whether that's unambiguously good depends on what happened to every *other* metric being tracked at the same time. This is the same regression-testing concern raised earlier: LLM-based systems are prone to exactly this pattern, where pushing one specific metric up quietly drags another one down. A genuine improvement means the metric being optimized went up **and** nothing else that was previously fine got worse in the process — checking only the one number that improved is exactly how a regression slips through unnoticed.